# Direct Data Sharing (Secure Shares)

Snowflake's **Direct Data Sharing** (also called Secure Shares) enables **zero-copy** data sharing between accounts.
The provider creates a **share** object, grants access to specific database objects, and adds consumer accounts.
Consumers then create a **database from the share** to query the data — no data is copied or transferred.

**Key characteristics:**
- Zero-copy: consumers read directly from the provider's storage
- Same-region: provider and consumer must be in the same cloud region
- Real-time: consumers always see the latest data
- Secure views are mandatory for sharing views

## 1. Setup — Database, Schema, Roles

We create a dedicated database (`sharing_demo`) with two schemas:
- `raw_data` — holds the source tables
- `shared_data` — holds secure views exposed to consumers

We also create two custom roles:
- `share_admin` — can create and manage shares
- `share_monitor` — can view share metadata and usage

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE sharing_demo;
CREATE OR REPLACE SCHEMA sharing_demo.shared_data;
CREATE OR REPLACE SCHEMA sharing_demo.raw_data;

CREATE ROLE IF NOT EXISTS share_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE share_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE share_admin;

GRANT CREATE SHARE ON ACCOUNT TO ROLE share_admin;
GRANT MANAGE SHARE TARGET ON ACCOUNT TO ROLE share_admin;
GRANT USAGE ON DATABASE sharing_demo TO ROLE share_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE sharing_demo TO ROLE share_admin;
GRANT CREATE VIEW ON SCHEMA sharing_demo.shared_data TO ROLE share_admin;
GRANT SELECT ON ALL TABLES IN SCHEMA sharing_demo.raw_data TO ROLE share_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

## 2. Create Sample Data

Using `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` as the source to populate our demo tables.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE sharing_demo;
USE SCHEMA raw_data;

CREATE OR REPLACE TABLE orders AS
SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate, o_orderpriority
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 10000;

CREATE OR REPLACE TABLE customers AS
SELECT c_custkey, c_name, c_nationkey, c_mktsegment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
LIMIT 5000;

## 3. Create Secure Views

**Secure views are mandatory for shares.** Regular views cannot be added to a share.
Secure views hide the view definition from consumers, preventing them from seeing the underlying query logic.

You can use `CURRENT_ACCOUNT()` inside a secure view to implement per-consumer row-level filtering.

In [ ]:
USE ROLE share_admin;
USE DATABASE sharing_demo;
USE SCHEMA shared_data;

CREATE OR REPLACE SECURE VIEW v_orders AS
SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate
FROM sharing_demo.raw_data.orders;

CREATE OR REPLACE SECURE VIEW v_customer_orders AS
SELECT
    c.c_custkey,
    c.c_name,
    c.c_mktsegment,
    o.o_orderkey,
    o.o_totalprice,
    o.o_orderdate
FROM sharing_demo.raw_data.customers c
JOIN sharing_demo.raw_data.orders o ON c.c_custkey = o.o_custkey;

## 4. Create and Configure the Share

Create the share object and grant access to specific database objects.
The grant hierarchy is: **database → schema → objects** (views/tables).

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SHARE demo_analytics_share
    COMMENT = 'Demo: Order analytics for partner access';

GRANT USAGE ON DATABASE sharing_demo TO SHARE demo_analytics_share;
GRANT USAGE ON SCHEMA sharing_demo.shared_data TO SHARE demo_analytics_share;
GRANT SELECT ON VIEW sharing_demo.shared_data.v_orders TO SHARE demo_analytics_share;
GRANT SELECT ON VIEW sharing_demo.shared_data.v_customer_orders TO SHARE demo_analytics_share;

## 5. Verify Share Configuration

In [ ]:
DESCRIBE SHARE demo_analytics_share;

SHOW GRANTS TO SHARE demo_analytics_share;

## 6. Add Consumer Account

> **Note:** Replace `<ORG_NAME>.<ACCOUNT_NAME>` with the actual consumer account identifier.
> This step requires knowing the target account.

In [ ]:
-- Uncomment and replace with actual consumer account:
-- ALTER SHARE demo_analytics_share ADD ACCOUNTS = <ORG_NAME>.<ACCOUNT_NAME>;

-- To verify accounts added:
SHOW SHARES LIKE 'DEMO_ANALYTICS_SHARE';

## 7. Test with Simulated Consumer

Use `SIMULATED_DATA_SHARING_CONSUMER` to test how the share appears to a consumer.
This session parameter lets you preview the consumer experience without an actual external account.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Simulate consumer view (replace with a valid account if needed)
-- ALTER SESSION SET SIMULATED_DATA_SHARING_CONSUMER = '<ORG_NAME>.<ACCOUNT_NAME>';

SELECT * FROM sharing_demo.shared_data.v_orders LIMIT 10;

SELECT * FROM sharing_demo.shared_data.v_customer_orders LIMIT 10;

-- Reset simulation
-- ALTER SESSION UNSET SIMULATED_DATA_SHARING_CONSUMER;

## 8. Monitoring

Query sharing metadata and usage views to track share activity.

In [ ]:
USE ROLE share_monitor;

-- List all outbound shares
SHOW SHARES;

-- Share history (last 7 days)
SELECT *
FROM TABLE(INFORMATION_SCHEMA.SHARE_HISTORY(
    DATE_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP()),
    DATE_RANGE_END => CURRENT_TIMESTAMP()
));

-- Databases created from shares (consumer side would run this)
SELECT database_name, origin, created_on
FROM SNOWFLAKE.ACCOUNT_USAGE.DATABASES
WHERE origin IS NOT NULL
ORDER BY created_on DESC;

## 9. Teardown

Clean up all objects created in this notebook.

In [ ]:
USE ROLE ACCOUNTADMIN;

DROP SHARE IF EXISTS demo_analytics_share;
DROP DATABASE IF EXISTS sharing_demo;
DROP ROLE IF EXISTS share_admin;
DROP ROLE IF EXISTS share_monitor;

---

**References:**
- [Secure Data Sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)
- [CREATE SHARE](https://docs.snowflake.com/en/sql-reference/sql/create-share)
- [Secure Views](https://docs.snowflake.com/en/user-guide/views-secure)